In [1]:
import os
import glob
import re
import rasterio
import geopandas as gpd
from shapely.geometry import Point

# 1. Map your directory pathways
base_yolo_dir = r"C:\Users\user\Downloads\python_projects\yolo_dataset"
# Point this to your original folder holding the TRUE georeferenced .tif tiles!
original_tif_dir = r"C:\Users\user\Downloads\python_projects\yolo_dataset\images" 

output_geojson_path = r"C:\Users\user\Downloads\python_projects\predicted_water_meters_georeferenced.geojson"
TARGET_CRS = "EPSG:32630" # UTM Zone 30N (Ghana)

dataset_splits = ["train", "valid", "test"]
meter_features = []
unmatched_tiles = set()

print("Recalibrating spatial grid using original GeoTIFF metadata...")

# 2. Iterate through all labels
for split in dataset_splits:
    label_sub_dir = os.path.join(base_yolo_dir, split, "labels")
    if not os.path.exists(label_sub_dir):
        continue
        
    label_files = glob.glob(os.path.join(label_sub_dir, "*.txt"))
    
    for txt_path in label_files:
        filename = os.path.basename(txt_path)
        
        # CLEANSE FILENAME: Strip Roboflow's added tags to find the original tile name
        # Roboflow usually appends '_tif_rf...' or changes the extension pattern.
        # This regex looks for your original tile pattern 'tile_y[digits]_x[digits]'
        match = re.search(r"(tile_y\d+_x\d+)", filename)
        if match:
            original_tile_name = match.group(1) + ".tif"
        else:
            # Fallback if your naming convention was different
            original_tile_name = filename.split('_tif')[0] + ".tif"
            
        tif_path = os.path.join(original_tif_dir, original_tile_name)
        
        # If it's not found in your central images folder, check the parent directory or common workspace
        if not os.path.exists(tif_path):
            unmatched_tiles.add(original_tile_name)
            continue
            
        # Open the TRUE georeferenced TIF file to get the real transform matrix
        with rasterio.open(tif_path) as src:
            transform = src.transform
            img_w, img_h = src.width, src.height
            
            # Read bounding box calculations
            with open(txt_path, "r") as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                        
                    class_id, x_center, y_center, bbox_w, bbox_h = map(float, parts)
                    
                    # Turn percentages back into physical pixel dimensions
                    pixel_x = x_center * img_w
                    pixel_y = y_center * img_h
                    
                    # Convert pixel offsets into real UTM coordinates using the true TIF matrix
                    geo_x, geo_y = transform * (pixel_x, pixel_y)
                    
                    meter_features.append({
                        "geometry": Point(geo_x, geo_y),
                        "class": "georeferenced_meter_point",
                        "split": split,
                        "parent_tile": original_tile_name
                    })

# 3. Print verification summaries
print(f"\n--- RECALIBRATION SUMMARY ---")
if unmatched_tiles:
    print(f"⚠️ Missing reference files: The script couldn't find {len(unmatched_tiles)} original .tif tiles in your image directory.")
    print(f"   Example missing file name looked for: {list(unmatched_tiles)[0]}")

if meter_features:
    gdf = gpd.GeoDataFrame(meter_features, crs=TARGET_CRS)
    gdf.to_file(output_geojson_path, driver="GeoJSON")
    print(f"🚀 Victory! Successfully georeferenced and compiled {len(gdf)} water meters.")
    print(f"True spatial layer saved to: {output_geojson_path}")
else:
    print("\n❌ Failed to georeference points. Your original .tif files are not inside your designated folder path.")


Recalibrating spatial grid using original GeoTIFF metadata...

--- RECALIBRATION SUMMARY ---
🚀 Victory! Successfully georeferenced and compiled 6128 water meters.
True spatial layer saved to: C:\Users\user\Downloads\python_projects\predicted_water_meters_georeferenced.geojson
